# Análisis de Distribución de Establecimientos Educativos y Bibliotecas Populares en Argentina

**Laboratorio de Datos - FCEyN (UBA)** **Integrantes:** Maite Alice, Abril Echarri y Facundo Miguel.

## Resumen
El objetivo principal de este trabajo práctico es investigar si existe algún patrón o relación entre la cantidad de Establecimientos Educativos (EE) y la de Bibliotecas Populares (BP) en las diferentes provincias y departamentos de Argentina, usando datos oficiales del censo 2022. Para ello seguimos los siguientes pasos:
1. **Revisión y limpieza de datos:** Examinamos la calidad de los datasets aplicando el enfoque GQM (Goal-Question-Metric).
2. **Modelado:** Diseñamos un DER en 3FN y su correspondiente modelo relacional.
3. **Preparación:** Limpiamos y creamos las respectivas tablas guiándonos de nuestro modelo relacional usando Python.
4. **Consultas y visualizaciones:** Mediante SQL y Matplotlib obtuvimos tablas que resumen cantidades y visualizaciones que muestran la distribución.

## Introducción
Partimos de datos oficiales del año 2022 y empezamos revisando qué tan completa y confiable es la información. Revisamos la calidad de los datos aplicando la técnica GQM para cuantificar la gravedad de los problemas detectados en las formas normales y completitud.

In [ ]:
import pandas as pd
import duckdb as dd
import matplotlib.pyplot as plt

# Cargamos las tablas que nos pasan por consigna
Establecimientos_educativos = pd.read_excel('../data/raw/2022_padron_oficial_establecimientos_educativos.xlsx', skiprows=6)
Bibliotecas_populares = pd.read_csv('../data/raw/bibliotecas-populares.csv')
censo = pd.read_excel('../data/raw/padron_poblacion.xlsX', skiprows=12) #se saltean las primeras 12 filas pues no contienen información relevante
censo = censo.loc[:56583]

## Revisión de la Calidad de Datos

### Problema 1: Inconsistencia en Establecimientos Educativos (EE)
Analizando la tabla original, notamos que no cumple con la Primera Forma Normal (1FN) ya que contiene atributos no atómicos (por ejemplo, múltiples teléfonos o extensiones en una misma celda).

* **Atributo de calidad afectado:** Consistencia.
* **Goal (Objetivo):** Asegurar la consistencia en los datos de la columna Teléfono.
* **Question (Pregunta):** ¿Cuál es la proporción de teléfonos inconsistentes en la columna teléfono?
* **Metric (Métrica):** `(Cantidad de teléfonos inconsistentes / Cantidad total de teléfonos) * 100`

Calculamos esta métrica a continuación:

In [ ]:
# Seleccionar la columna de teléfono y convertirla a string (renombramos por claridad)
Establecimientos_educativos['Teléfono'] = Establecimientos_educativos['Teléfono'].astype(str).str.strip()

# Identificar registros inconsistentes
inconsistentes = Establecimientos_educativos[
    (Establecimientos_educativos['Teléfono'].str.len() != 9) |
    (Establecimientos_educativos['Teléfono'].str[4] != '-') |
    (Establecimientos_educativos['Teléfono'].str.contains(r'[^0-9-]', regex=True))
]

# Calcular porcentaje de inconsistencias
porcentaje_inconsistentes = 100 * len(inconsistentes) / len(Establecimientos_educativos)

print(f"Porcentaje de teléfonos inconsistentes: {porcentaje_inconsistentes:.2f}%")

### Problema 2: Completitud en Modalidad Común (EE)
Dentro del campo 'Común' (y sus subcolumnas: Jardín maternal, Jardín de infantes, Primario, Secundario, etc.) se detecta una gran cantidad de valores nulos, generando dudas sobre si significan "no aplica", "no disponible" o "error de carga".

* **Atributo de calidad afectado:** Completitud e Interpretabilidad.
* **Goal (Objetivo):** Reducir la cantidad de valores nulos en la columna 'Modalidad' para mejorar la calidad de la información.
* **Question (Pregunta):** ¿Cuál es el porcentaje de valores nulos en la columna 'Modalidad'?
* **Metric (Métrica):** `(Cantidad de valores nulos / Cantidad Total de valores) * 100`

In [ ]:
# Reemplazar strings vacíos o con solo espacios por None (NaN)
Establecimientos_educativos['Común'] = Establecimientos_educativos['Común'].replace(r'^\s*$', None, regex=True)

# Total de filas
total = len(Establecimientos_educativos)

# Contar valores no nulos y no vacíos (completos)
completos = Establecimientos_educativos['Común'].notnull().sum()

# Calcular porcentaje de incompletitud
porcentaje_incompletos = 100 * (total - completos) / total

print(f"Porcentaje de valores incompletos en la columna 'Común': {porcentaje_incompletos:.2f}%")

### Problema 3: Completitud en Bibliotecas Populares (BP)
La tabla de BP tampoco cumple con la 1FN ni la 3FN, conteniendo varios campos vacíos o no atómicos. 

* **Atributo de calidad afectado:** Completitud.
* **Goal (Objetivo):** Asegurar la completitud de los datos en la columna Mail para garantizar un análisis representativo.
* **Question (Pregunta):** ¿Cuál es la proporción de registros con valor nulo en la columna Mail?
* **Metric (Métrica):** `(Cantidad de valores nulos / Cantidad Total de valores) * 100`

In [ ]:
def metrica_GQM_mail(Bibliotecas_populares):
    nulls = Bibliotecas_populares['mail'].isna().sum() 
    total = len(Bibliotecas_populares)
    return (nulls * 100) / total

resultado = metrica_GQM_mail(Bibliotecas_populares)
print(f"Porcentaje de valores nulos o vacíos en la columna 'mail': {resultado:.2f}%")

## Procesamiento y Limpieza de Datos

Para modelar los datos, diseñamos un esquema relacional con las tablas: `Población`, `EE`, `BP`, `Departamento`, `Niveles` y la relación `Esta_formada_por`.

### 1. Tabla Población
Agrupamos las edades según el grupo etario (jardín, primario, secundario, total) para facilitar nuestro análisis. 
* **Decisiones tomadas:** Unificamos las comunas con `id_depto = 2000` tratando a CABA como un único departamento. Además, modificamos los IDs correspondientes a Río Grande y Ushuaia para que coincidan con las otras tablas.

In [ ]:
def separar_por_niveles(df_edades):
    grupo_etario = {'jardin': 0, 'primaria': 0, 'secundaria': 0, 'total': 0}

    for _, row in df_edades.iterrows():
        cantidad = row['Cantidad']
        grupo = row['Edad']
        grupo_etario['total'] += cantidad
        if 0 <= grupo <= 5:
            grupo_etario['jardin'] += cantidad
        elif 6 <= grupo <= 12:
            grupo_etario['primaria'] += cantidad
        elif 13 <= grupo <= 18:
            grupo_etario['secundaria'] += cantidad
            
    nuevas_filas = [
        {'Grupo_Etario':'jardin', 'Cantidad' : grupo_etario['jardin']},
        {'Grupo_Etario':'primaria','Cantidad' : grupo_etario['primaria']},
        {'Grupo_Etario':'secundaria','Cantidad' : grupo_etario['secundaria']},
        {'Grupo_Etario':'total', 'Cantidad' : grupo_etario['total']}
    ]
    
    df_nivel_educativo = df_edades.rename(columns={'Edad': 'Grupo_Etario'}).iloc[0:0]                                                          
    return pd.concat([df_nivel_educativo, pd.DataFrame(nuevas_filas)])

def crear_tabla_poblacion(censo):
    Poblacion = pd.DataFrame(columns=['id_depto', 'Depto', 'Grupo_Etario', 'Cantidad'])
    comuna_caba_acumulada = pd.DataFrame(columns=['Edad', 'Cantidad'])

    for fila in range(len(censo)):
        valor_columna = str(censo.loc[fila].iloc[1])
        if "AREA" in valor_columna:
            dato_depto = valor_columna
            codigo_depto = int(dato_depto.split(" ")[2])
            nombre_depto = str(censo.loc[fila].iloc[2])
            es_comuna_caba = 'Comuna' in nombre_depto
            if es_comuna_caba:
                codigo_depto, nombre_depto = 2000, 'Ciudad Autónoma de Buenos Aires'
            df_edades = pd.DataFrame(columns=['Edad', 'Cantidad'])
        elif not pd.isna(censo.loc[fila].iloc[1]) and not isinstance(censo.loc[fila].iloc[1], str):
            df_edades.loc[len(df_edades)] = [int(censo.loc[fila].iloc[1]), int(censo.loc[fila].iloc[2])]

        if (fila + 1 == len(censo)) or ("AREA" in str(censo.loc[fila + 1].iloc[1])):
            if 'df_edades' in locals() and not df_edades.empty:
                if es_comuna_caba:
                    comuna_caba_acumulada = pd.concat([comuna_caba_acumulada, df_edades])
                else:
                    df_niveles = separar_por_niveles(df_edades)
                    for _, row in df_niveles.iterrows():
                        Poblacion.loc[len(Poblacion)] = [codigo_depto, nombre_depto, row['Grupo_Etario'], row['Cantidad']]

    if not comuna_caba_acumulada.empty:
        df_niveles_caba = separar_por_niveles(comuna_caba_acumulada.groupby('Edad', as_index=False).sum())
        for _, row in df_niveles_caba.iterrows():
            Poblacion.loc[len(Poblacion)] = [2000, 'Ciudad Autónoma de Buenos Aires', row['Grupo_Etario'], row['Cantidad']]
    return Poblacion

poblacion = crear_tabla_poblacion(censo)
poblacion['id_depto'] = poblacion['id_depto'].replace({94008: 94007, 94015: 94014})

### 2. Tabla Establecimientos Educativos (EE)
* **Decisiones tomadas:** Nos quedamos solamente con los datos correspondientes a la modalidad 'Común'. Renombramos la provincia de 'Ciudad de Buenos Aires' a 'Ciudad Autónoma de Buenos Aires' para mantener la consistencia con la tabla de Bibliotecas Populares.

In [ ]:
def crear_tabla_EE(EE):
    EE = EE[EE['Común'] == 1].copy()
    def obtener_id_depto(fila):
        if fila['Jurisdicción'] == 'Ciudad de Buenos Aires': return 2000
        id_depto = str(fila['Código de localidad']).lstrip('0')
        return int(id_depto[:4]) if fila['Jurisdicción'] == 'Buenos Aires' else int(id_depto[:5])

    EE['id_depto'] = EE.apply(obtener_id_depto, axis=1)
    EE.rename(columns={'Jurisdicción': 'Provincia'}, inplace=True)
    EE['Provincia'] = EE['Provincia'].replace('Ciudad de Buenos Aires', 'Ciudad Autónoma de Buenos Aires')
    return EE[['Cueanexo', 'Nombre', 'id_depto', 'Provincia']]

ee = crear_tabla_EE(Establecimientos_educativos)

### 3. Tabla Bibliotecas Populares (BP)
* **Decisiones tomadas:** Agregamos un 'Índice' para usar como clave primaria. Nos quedamos únicamente con los dominios de la columna `mail`, ya que es lo relevante para el análisis, rellenando los nulos con `sin_dominio`. A las fechas de fundación nulas se les asignó `sin fecha`.

In [ ]:
bp = dd.sql("""
    SELECT fecha_fundacion, nombre, id_departamento AS id_depto, mail, provincia
    FROM Bibliotecas_populares
""").df()

bp.insert(0, 'Indice', range(1, len(bp) + 1))
bp['dominio'] = bp['mail'].str.extract(r'@([a-zA-Z0-9\-]+)\.').fillna('sin_dominio').str.lower() 
bp['fecha_fundacion'] = bp['fecha_fundacion'].fillna('sin fecha').astype(str)
bp = bp.drop(columns=['mail'])

### 4. Tabla Departamento
* **Decisiones tomadas:** Completamos manualmente algunos datos faltantes. Por ejemplo, asignamos la provincia al departamento Tolhuin, ya que figuraba en la tabla de población pero no aparecía ni en BP ni en EE.
* *Nota:* En la base original Tierra del Fuego tiene cuatro departamentos, pero en el Censo 2022 no se consideró a Antártida Argentina. Por eso, nuestra base finaliza con 513 departamentos en total.

In [ ]:
def crear_tabla_departamento(poblacion, ee, bp):
    sin_repetidos_poblacion = poblacion[['id_depto', 'Depto']].drop_duplicates().rename(columns={'Depto': 'Nombre'})
    sin_repetidos_ee = ee[['id_depto', 'Provincia']].drop_duplicates()
    sin_repetidos_bp = bp[['id_depto', 'provincia']].drop_duplicates().rename(columns={'provincia': 'Provincia'})

    union_ee_bp_sin_repetidos = dd.sql("""
        SELECT DISTINCT * FROM (SELECT * FROM sin_repetidos_ee UNION SELECT * FROM sin_repetidos_bp)
    """).df()
                    
    departamento = dd.sql("""
        SELECT p.id_depto, p.Nombre, u.Provincia
        FROM sin_repetidos_poblacion AS p
        LEFT JOIN union_ee_bp_sin_repetidos AS u ON p.id_depto = u.id_depto
    """).df()
    departamento['Provincia'] = departamento['Provincia'].mask(departamento['id_depto']==94011, 'Tierra del Fuego')
    return departamento

departamento = crear_tabla_departamento(poblacion, ee, bp)

# Limpieza final de columnas redundantes
poblacion = poblacion[['id_depto', 'Grupo_Etario', 'Cantidad']]
bp = bp[['Indice', 'id_depto', 'fecha_fundacion', 'nombre', 'dominio']]
ee = ee[['Cueanexo', 'id_depto','Nombre']]

### 5. Tablas Niveles y Relación "Está Formada Por"
* **Decisiones tomadas:** Unificamos los niveles educativos. Agrupamos jardín maternal e infantes en `jardin`, primario queda igual, y agrupamos secundario e INET en `secundario`. Excluimos los niveles 'SNU' por no poder asociarlos a un grupo etario.

Procedemos a armar estas tablas y guardar nuestro modelo procesado en formato CSV.

In [ ]:
niveles = pd.DataFrame({"Sector" : ["jardin", "primaria", "secundaria"]})

def crear_esta_formada_por(EE_original):
    tabla_ee = EE_original[["Nivel inicial - Jardín maternal", "Nivel inicial - Jardín de infantes", "Primario", "Secundario", "Secundario - INET", "Cueanexo"]]
    esta_formada_por = pd.DataFrame(columns=["Cueanexo_EE", "Sector_niveles"])

    for i in range(len(tabla_ee)):
        cue = tabla_ee.loc[i, "Cueanexo"]
        if tabla_ee.loc[i, "Nivel inicial - Jardín maternal"] == 1 or tabla_ee.loc[i, "Nivel inicial - Jardín de infantes"] == 1:
            esta_formada_por.loc[len(esta_formada_por)] = [cue, "jardin"]
        if tabla_ee.loc[i, "Primario"] == 1:
            esta_formada_por.loc[len(esta_formada_por)] = [cue, "primario"]
        if tabla_ee.loc[i, "Secundario"] == 1 or tabla_ee.loc[i, "Secundario - INET"] == 1:
            esta_formada_por.loc[len(esta_formada_por)] = [cue, "secundario"]
    return esta_formada_por

esta_formada_por = crear_esta_formada_por(Establecimientos_educativos)

# Exportar a CSV
esta_formada_por.to_csv("../data/processed/esta_formada_por.csv", index=False)
poblacion.to_csv("../data/processed/poblacion.csv", index=False)
niveles.to_csv("../data/processed/niveles.csv", index=False)
ee.to_csv("../data/processed/ee.csv", index=False)
departamento.to_csv("../data/processed/departamento.csv", index=False)
bp.to_csv("../data/processed/bp.csv", index=False)

## Análisis de Datos: Consultas SQL y Visualizaciones

### Consigna I
> Para cada departamento informar la provincia, el nombre del departamento, la cantidad de EE de cada nivel educativo (modalidad común), y la cantidad de habitantes por edad según los niveles educativos. Ordenado alfabéticamente por provincia y, dentro de estas, descendente por escuelas primarias.

In [ ]:
consigna_1 = dd.sql (""" 
    SELECT d.Provincia, d.Nombre AS Departamento,
           COUNT(CASE WHEN efp.Sector_niveles = 'jardin' THEN 1 END) AS Jardines,
           MAX(CASE WHEN p.Grupo_Etario = 'jardin' THEN p.Cantidad ELSE 0 END) AS Poblacion_Jardin,
           COUNT(CASE WHEN efp.Sector_niveles = 'primario' THEN 1 END) AS Primarias,
           MAX(CASE WHEN p.Grupo_Etario = 'primaria' THEN p.Cantidad ELSE 0 END) AS Poblacion_Primaria,
           COUNT(CASE WHEN efp.Sector_niveles = 'secundario' THEN 1 END) AS Secundarios,
           MAX(CASE WHEN p.Grupo_Etario = 'secundaria' THEN p.Cantidad ELSE 0 END) AS Poblacion_Secundaria
    FROM departamento AS d
    LEFT JOIN ee ON ee.id_depto = d.id_depto
    LEFT JOIN esta_formada_por AS efp ON efp.cueanexo_EE = ee.cueanexo
    LEFT JOIN poblacion AS p ON p.id_depto = d.id_depto
    GROUP BY d.Provincia, d.Nombre
    ORDER BY d.Provincia ASC, Primarias DESC;
""").df()
consigna_1.head()

### Gráfico 1: Cantidad de BP por provincia

In [ ]:
grafico1_data = dd.sql("""
    SELECT d.provincia, COUNT (*) as cantidad
    FROM departamento as d 
    LEFT JOIN bp ON d.id_depto = bp.id_depto
    GROUP BY d.provincia
    ORDER BY cantidad DESC
""").df()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(grafico1_data['Provincia'][::-1], grafico1_data['cantidad'][::-1], color='darkturquoise')
ax.set_xlabel('Cantidad de Bibliotecas Populares')
ax.set_title('Cantidad de BP por Provincia')
ax.grid(True, linestyle="--", alpha=0.5)
plt.show()

**Conclusión del Gráfico 1:** Se observa claramente que Buenos Aires encabeza la lista con la mayor cantidad de BP, seguida de Santa Fe y Córdoba. En el otro extremo, Tierra del Fuego tiene apenas 10 bibliotecas.

---

### Gráfico 2: EE vs. Población (Por nivel educativo)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(data=consigna_1, x='Poblacion_Jardin', y='Jardines', s=6, color='red', alpha=0.3, label="jardin")
ax.scatter(data=consigna_1, x='Poblacion_Primaria', y='Primarias', s=6, color='darkturquoise', alpha=0.3, label="primaria")
ax.scatter(data=consigna_1, x='Poblacion_Secundaria', y='Secundarios', s=6, color='blue', alpha=0.2, label="secundaria")
ax.set_xlim(0, 60000); ax.set_ylim(0, 800)
ax.legend(title="Niveles Educativos")
ax.set_title("Cantidad de EE en función de la población")
plt.show()

In [ ]:
**Conclusión del Gráfico 2:** Se observa una tendencia creciente: a medida que aumenta la población por nivel educativo, también aumenta la cantidad de EE, lo cual es esperable. Sin embargo, hay una alta concentración de puntos en valores bajos. También se nota que los establecimientos de nivel jardín y primaria son más numerosos que los de secundaria, posiblemente reflejando una mayor cobertura inicial.

---

### Consigna II
> Para cada departamento informar la provincia, el nombre y la cantidad de BP fundadas desde 1950. Ordenado alfabéticamente por provincia y descendente por cantidad de BP.

In [ ]:
def tabla_2():
    return dd.sql("""
        SELECT d.Provincia, d.Nombre AS Departamento,
            COUNT(CASE WHEN bp.fecha_fundacion >= '1950-01-01' THEN 1 ELSE NULL END) AS Cantidad_BP_Fundadas_Desde_1950
        FROM departamento d
        LEFT JOIN bp ON bp.id_depto = d.id_depto
        GROUP BY d.Provincia, d.Nombre
        ORDER BY d.Provincia ASC, Cantidad_BP_Fundadas_Desde_1950 DESC;
    """).df()
                    
consigna_2 = tabla_2()
consigna_2.head()

### Consigna III
> Para cada departamento, indicar provincia, nombre del departamento, cantidad de BP, cantidad de EE (modalidad común) y población total. Ordenar por cantidad EE descendente, cantidad BP descendente y nombres ascendentes.

In [ ]:
def tabla_3():
    ee_por_depto = dd.sql("SELECT id_depto, COUNT(*) AS Cantidad_EE FROM ee GROUP BY id_depto").df()
    bp_por_depto = dd.sql("SELECT id_depto, COUNT(*) AS Cantidad_BP FROM bp GROUP BY id_depto").df()
    poblacion_total = dd.sql("SELECT id_depto, MAX(Cantidad) AS Poblacion FROM poblacion WHERE Grupo_Etario = 'total' GROUP BY id_depto").df()

    return dd.sql("""
        SELECT d.Provincia, d.Nombre AS Departamento,
            CASE WHEN ee.Cantidad_EE IS NULL THEN 0 ELSE ee.Cantidad_EE END AS Cantidad_EE,
            CASE WHEN bp.Cantidad_BP IS NULL THEN 0 ELSE bp.Cantidad_BP END AS Cantidad_BP,
            CASE WHEN p.Poblacion IS NULL THEN 0 ELSE p.Poblacion END AS Poblacion
        FROM departamento AS d
        LEFT JOIN bp_por_depto AS bp ON d.id_depto = bp.id_depto
        LEFT JOIN ee_por_depto AS ee ON d.id_depto = ee.id_depto
        LEFT JOIN poblacion_total AS p ON d.id_depto = p.id_depto
        ORDER BY Cantidad_EE DESC, Cantidad_BP DESC, d.Provincia ASC, d.Nombre ASC
    """).df()

consigna_3 = tabla_3()
consigna_3.head()

### Consigna IV
> Para cada departamento, indicar provincia, nombre del departamento y qué dominios de mail se usan más para las BP.

*Nota:* Si en un departamento hay empate en la cantidad máxima de un dominio, se muestran todos los dominios empatados.

In [ ]:
def tabla_4():
    dominio_por_depto = dd.sql("SELECT id_depto, dominio, COUNT(*) AS cantidad FROM bp GROUP BY id_depto, dominio").df()
  
    maximos = dd.sql("""
        SELECT d1.id_depto, d1.dominio, d1.cantidad
        FROM dominio_por_depto AS d1
        WHERE d1.cantidad >= ALL (
            SELECT d2.cantidad
            FROM dominio_por_depto AS d2
            WHERE d1.id_depto = d2.id_depto
        )
    """).df()
    
    return dd.sql("""
        SELECT d.nombre AS Departamento, d.provincia AS Provincia, m.dominio AS Dominio_mas_frecuente_en_BP
        FROM maximos AS m
        LEFT JOIN departamento AS d ON m.id_depto = d.id_depto
        ORDER BY d.nombre, d.provincia
    """).df()

consigna_4 = tabla_4()
consigna_4.head()

### Gráfico 3: Distribución de EE por Departamento (Boxplot por Provincia)

In [ ]:
def grafico3():
    return dd.sql("""
        SELECT d.provincia, d.Nombre AS depto ,COUNT (*) as cantidad
        FROM departamento as d 
        LEFT JOIN ee ON d.id_depto = ee.id_depto
        GROUP BY d.provincia, d.Nombre
        ORDER BY cantidad DESC
    """).df()

grafico3_data = grafico3()

# Ordenamos las provincias según la mediana de EE por departamento
orden = grafico3_data.groupby('Provincia')['cantidad'].median().sort_values().index

# Crear lista de datos por provincia
datos = [grafico3_data[grafico3_data['Provincia'] == prov]['cantidad'].values for prov in orden]
 
plt.figure(figsize=(10, 6))
plt.boxplot(datos, labels=orden, patch_artist=True,
            boxprops=dict(facecolor='skyblue'),
            medianprops=dict(color='red'))

plt.title('Distribución de EE por Departamento (por Provincia)')
plt.xlabel('Provincia')
plt.ylabel('Cantidad de EE por Departamento')
plt.xticks(rotation=45, ha='right') 
plt.grid(True, linestyle='--', alpha=0.5) 
plt.show()

**Conclusión del Gráfico 3:** Se observa una alta variabilidad entre provincias. CABA presenta un valor atípico muy elevado. También hay muchos outliers en provincias como Buenos Aires y Córdoba, lo que indica que, aunque la mediana parezca similar, la distribución no es homogénea (hay departamentos con muchísimos EE y otros con muy pocos). Por otro lado, provincias de menor densidad como Santa Cruz, La Pampa o Chubut muestran una distribución más homogénea (rangos cortos y medianas bajas).

---

### Gráfico 4: Relación entre BP y EE (cada mil habitantes)

In [ ]:
# Hacemos una copia del DataFrame de la consigna 3
grafico4 = consigna_3.copy()

grafico4['Poblacion'] = grafico4['Poblacion'] / 1000
grafico4['Cantidad_BP'] = grafico4['Cantidad_BP'] / grafico4['Poblacion']
grafico4['Cantidad_EE'] = grafico4['Cantidad_EE'] / grafico4['Poblacion']
grafico4 = grafico4[['Cantidad_BP', 'Cantidad_EE']]

fig, ax = plt.subplots()
ax.scatter(data=grafico4, x='Cantidad_BP', y='Cantidad_EE', s=3, color="green", edgecolors='gray', linewidths=0.2)
ax.set_title("Relación entre la cantidad de BP y EE cada mil habitantes")
ax.set_xlabel("Cantidad de Biblotecas Populares (cada mil habitantes)")
ax.set_ylabel("Cantidad de Establecimientos Educativos \n (cada mil habitantes)")
plt.show()

**Conclusión del Gráfico 4:** Si bien no vemos una relación muy marcada, es evidente que a menor cantidad de BP, existe una menor cantidad de EE. Hay muchos casos de departamentos que tienen muy pocas Bibliotecas Populares y distintas cantidades de Establecimientos Educativos, demostrando una alta dispersión de datos.

## Conclusión Final del Trabajo
A partir del análisis realizado en este informe, podemos concluir que **no existe una relación directa** entre la cantidad de Establecimientos Educativos y la de Bibliotecas Populares. 

Si bien se observa una ligera tendencia creciente en función de la población, la amplia dispersión de los datos y el elevado número de departamentos con baja densidad de bibliotecas, pero con variaciones significativas en la cantidad de establecimientos educativos, impiden establecer una correlación sólida entre ambos indicadores. 

Deducimos que un elevado número de EE por cada mil habitantes no asegura una alta densidad de BP, ni viceversa. Consideramos que esto se debe a otros aspectos que no estamos contemplando (factores culturales, financiamiento o gestión provincial). Posiblemente, al incluir variables adicionales de otras fuentes, podríamos llegar a conclusiones más precisas.